In [ ]:
"""XMA 22 signals to merged reporter"""


In [ ]:
# Snippet for resolving module paths correctly and allows us to run individual py files

# clear_output allows clearing cell output if we print something in a for loop
# useful if we want to push pnl updates or live-print performance on a chart during the backtest steps
from IPython.display import clear_output
import json
import pandas as pd
import os
import sys

module_path = os.path.abspath(os.path.join("../../backtester-api"))
if module_path not in sys.path:
    sys.path.append(module_path)


In [ ]:
# Report and indicator setup

import pathlib
fileFolderPath = pathlib.Path().resolve()

# --- Reports ---

# Show
SHOW_PLOT_GRAPHS_SINGLE = False
SHOW_PLOT_GRAPHS_MERGED = True

SHOW_HTML_REPORT_SINGLE = False
SHOW_HTML_REPORT_MERGED = False

# Save
SAVE_HTML_REPORT_SINGLE = True
SAVE_HTML_REPORT_MERGED = True
SAVE_HTML_REPORT_FOLDER = f'{fileFolderPath}/../temp/results'
SAVE_AS_DATASET = False
SAVE_DATASET_FOLDER = f'{fileFolderPath}/../datasets/tradingview'

# --- Indicators ---

signal_key = "XMA"

indicator_key = 'hma'
input_scale = 1
output_scale = 2

date_from = "2020-01-01"
date_to = "2022-01-01"

deposit_asset = "USD"
deposit_volume = 100000


In [ ]:
# local modules
from app.packages.backtester.backtester import Backtester
from app.packages.backtester.indicators import Indicators
from app.packages.backtester.market import MarketData, Market, MarketDataToTradingViewCSV
from app.packages.backtester.portfolio import WeightedSignal, Portfolio
from app.packages.backtester.signals.base import SignalSide
from app.packages.backtester.signals.ma import SignalMaCrossover
from app.packages.backtester.exchange import Exchange, MarginAllocationType, MarketType
from app.packages.backtester.reporter import Reporter


In [ ]:
# Markets
market = Market()


btc_symbol = "BTC-PERP/USD"
alt_symbol = "ALT-PERP/USD"
mid_symbol = "MID-PERP/USD"
shit_symbol = "SHIT-PERP/USD"

# Division of total budget between symbols
symbol_budgets = {}
symbol_budgets[btc_symbol] = 0.625
symbol_budgets[alt_symbol] = 0.125
symbol_budgets[mid_symbol] = 0.125
symbol_budgets[shit_symbol] = 0.125

benchmark_symbols = [btc_symbol]

market.add_market(
    symbol=btc_symbol,
    df=MarketData(symbol=btc_symbol, date_from=date_from, date_to=date_to,
                  interval=1, unit_of_time='hour').get_df()
)
market.add_market(
    symbol=alt_symbol,
    df=MarketData(symbol=alt_symbol, date_from=date_from, date_to=date_to,
                  interval=1, unit_of_time='hour').get_df()
)
market.add_market(
    symbol=mid_symbol,
    df=MarketData(symbol=mid_symbol, date_from=date_from, date_to=date_to,
                  interval=1, unit_of_time='hour').get_df()
)
market.add_market(
    symbol=shit_symbol,
    df=MarketData(symbol=shit_symbol, date_from=date_from, date_to=date_to,
                  interval=1, unit_of_time='hour').get_df()
)

# Setup candles
ohlc = {}
ohlc[btc_symbol] = {}
ohlc[alt_symbol] = {}
ohlc[mid_symbol] = {}
ohlc[shit_symbol] = {}

interval = 2
unit_of_time = 'hour'
ohlc_key = f'{interval}{unit_of_time}'
ohlc[btc_symbol][ohlc_key] = MarketData(symbol=btc_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()

interval = 6
unit_of_time = 'hour'
ohlc_key = f'{interval}{unit_of_time}'
ohlc[btc_symbol][ohlc_key] = MarketData(symbol=btc_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[alt_symbol][ohlc_key] = MarketData(symbol=alt_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[mid_symbol][ohlc_key] = MarketData(symbol=mid_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[shit_symbol][ohlc_key] = MarketData(symbol=shit_symbol, date_from=date_from,
                                         date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()

interval = 1
unit_of_time = 'day'
ohlc_key = f'{interval}{unit_of_time}'
ohlc[btc_symbol][ohlc_key] = MarketData(symbol=btc_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[alt_symbol][ohlc_key] = MarketData(symbol=alt_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[mid_symbol][ohlc_key] = MarketData(symbol=mid_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[shit_symbol][ohlc_key] = MarketData(symbol=shit_symbol, date_from=date_from,
                                         date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()

interval = 3
unit_of_time = 'day'
ohlc_key = f'{interval}{unit_of_time}'
ohlc[btc_symbol][ohlc_key] = MarketData(symbol=btc_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[alt_symbol][ohlc_key] = MarketData(symbol=alt_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[mid_symbol][ohlc_key] = MarketData(symbol=mid_symbol, date_from=date_from,
                                        date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()
ohlc[shit_symbol][ohlc_key] = MarketData(symbol=shit_symbol, date_from=date_from,
                                         date_to=date_to, interval=interval, unit_of_time=unit_of_time).get_df()

print(
    f'Budget reserved for {btc_symbol} is {deposit_volume * symbol_budgets[btc_symbol]} {deposit_asset}')
print(
    f'Budget reserved for {alt_symbol} is {deposit_volume * symbol_budgets[alt_symbol]} {deposit_asset}')
print(
    f'Budget reserved for {mid_symbol} is {deposit_volume * symbol_budgets[mid_symbol]} {deposit_asset}')
print(
    f'Budget reserved for {shit_symbol} is {deposit_volume * symbol_budgets[shit_symbol]} {deposit_asset}')

In [ ]:
class SignalSetup(object):
    def __init__(
        self,
        unit_of_time,
        symbol,
        side,
        interval,
        base_length,
        trend_length,
        base_key,
        trend_key,
        signal_key,
        sl_percent,
        tp_percent,
        ohlc,
        budget_scale=1,
        input_scale=1,
        output_scale=1
    ):
        self.unit_of_time = unit_of_time
        self.symbol = symbol
        self.side = side
        self.interval = interval
        self.base_length = base_length
        self.trend_length = trend_length
        self.base_key = base_key
        self.trend_key = trend_key
        self.signal_key = signal_key
        self.sl_percent = sl_percent
        self.tp_percent = tp_percent
        self.budget_scale = budget_scale
        self.output_scale = output_scale
        self.input_scale = input_scale
        self.ohlc = ohlc


signal_setup_list = []


In [ ]:
# ------- BTC-PERP ------
symbol = btc_symbol
# 2h long
trend_length = 70
base_length = 100
interval = 2
unit_of_time = 'hour'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.06,
    tp_percent=0.25,
    budget_scale=0.1,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 6h long
trend_length = 15
base_length = 78
interval = 6
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.1,
    tp_percent=0.25,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 6h short
trend_length = 50
base_length = 75
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.04,
    tp_percent=0.2,
    budget_scale=0.05,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 1d long
trend_length = 11
base_length = 64
interval = 1
unit_of_time = 'day'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.3,
    tp_percent=0.5,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 1d short
trend_length = 28
base_length = 73
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.1,
    tp_percent=0.2,
    budget_scale=0.05,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 3d long
trend_length = 10
base_length = 20
interval = 3
unit_of_time = 'day'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.25,
    tp_percent=1.25,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 3d short
trend_length = 12
base_length = 2
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.02,
    tp_percent=0.2,
    budget_scale=0.05,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))

print(f"{symbol} signals setup created!")


In [ ]:
# ------- ALT-PERP ------
symbol = alt_symbol
# 6h long
unit_of_time = 'hour'
trend_length = 10
base_length = 90
interval = 6
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.1,
    tp_percent=1.2,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 6h short
trend_length = 15
base_length = 80
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.06,
    tp_percent=0.2,
    budget_scale=0.1,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 1d long
trend_length = 8
base_length = 60
interval = 1
unit_of_time = 'day'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.1,
    tp_percent=1.5,
    budget_scale=0.3,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 1d short
trend_length = 24
base_length = 80
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.06,
    tp_percent=0.4,
    budget_scale=0.1,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 3d long
trend_length = 8
base_length = 29
interval = 3
unit_of_time = 'day'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.25,
    tp_percent=2.5,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))

print(f"{symbol} signals setup created!")


In [ ]:
# ------- MID-PERP ------
symbol = mid_symbol
# 6h long
unit_of_time = 'hour'
trend_length = 10
base_length = 90
interval = 6
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.1,
    tp_percent=1.2,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 6h short
trend_length = 15
base_length = 80
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.06,
    tp_percent=0.2,
    budget_scale=0.1,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 1d long
trend_length = 8
base_length = 60
interval = 1
unit_of_time = 'day'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.1,
    tp_percent=1.5,
    budget_scale=0.3,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 1d short
trend_length = 24
base_length = 80
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.06,
    tp_percent=0.4,
    budget_scale=0.1,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 3d long
trend_length = 8
base_length = 29
interval = 3
unit_of_time = 'day'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.25,
    tp_percent=2.5,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))

print(f"{symbol} signals setup created!")


In [ ]:
# ------- SHIT-PERP ------
symbol = shit_symbol
# 6h long
unit_of_time = 'hour'
trend_length = 10
base_length = 90
interval = 6
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.1,
    tp_percent=1.2,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 6h short
trend_length = 15
base_length = 80
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.06,
    tp_percent=0.2,
    budget_scale=0.1,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 1d long
trend_length = 8
base_length = 60
interval = 1
unit_of_time = 'day'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.1,
    tp_percent=1.5,
    budget_scale=0.3,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 1d short
trend_length = 24
base_length = 80
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.short,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.short.value}_{base_length}_{trend_length}',
    sl_percent=0.06,
    tp_percent=0.4,
    budget_scale=0.1,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))
# 3d long
trend_length = 8
base_length = 29
interval = 3
unit_of_time = 'day'
signal_setup_list.append(SignalSetup(
    unit_of_time=unit_of_time,
    symbol=symbol,
    side=SignalSide.long,
    interval=interval,
    base_length=base_length,
    trend_length=trend_length,
    base_key=f'{interval}{unit_of_time}_{indicator_key}_{base_length}',
    trend_key=f'{interval}{unit_of_time}_{indicator_key}_{trend_length}',
    signal_key=f'{signal_key}_{symbol}_{interval}{unit_of_time}_{SignalSide.long.value}_{base_length}_{trend_length}',
    sl_percent=0.25,
    tp_percent=2.5,
    budget_scale=0.25,
    output_scale=output_scale,
    input_scale=input_scale,
    ohlc=ohlc[symbol][f'{interval}{unit_of_time}']
))

print(f"{symbol} signals setup created!")


In [ ]:

for signal_setup in signal_setup_list:
    market.add_indicator(symbol=signal_setup.symbol, interval=signal_setup.interval, unit_of_time=signal_setup.unit_of_time,
                         indicator_name=signal_setup.trend_key, df=Indicators.hma(signal_setup.ohlc, signal_setup.trend_length))
    market.add_indicator(symbol=signal_setup.symbol, interval=signal_setup.interval, unit_of_time=signal_setup.unit_of_time,
                         indicator_name=signal_setup.base_key, df=Indicators.hma(signal_setup.ohlc, signal_setup.base_length))

# we need to compile market before feeding it into the exchange.
market.compile()

print("Market is ready!")


In [ ]:
class BacktestSetup(object):
    def __init__(self, key, signal_setup, portfolio, exchange, reporter, backtester=None):
        self.key = key
        self.signal_setup = signal_setup
        self.portfolio = portfolio
        self.exchange = exchange
        self.reporter = reporter
        self.backtester = backtester


backtest_setup_list = []


In [ ]:
# adding backtest setups
for signal_setup in signal_setup_list:
    portfolio = Portfolio(weighted_signals=[
        WeightedSignal(
            weight=signal_setup.input_scale,
            signal=SignalMaCrossover(key=signal_setup.signal_key, market=market, symbol=signal_setup.symbol, side=signal_setup.side,
                                     fast_indicator_key=signal_setup.trend_key, slow_indicator_key=signal_setup.base_key,
                                     sl_percent=signal_setup.sl_percent, tp_percent=signal_setup.tp_percent)
        )
    ], output_scale=signal_setup.output_scale)
    exchange = Exchange(
        market=market,
        slippage=0.002,
        maker_fee=0.00125,
        taker_fee=0.00075,
        market_type=MarketType.future,
        max_leverage=10,
        margin_allocation_type=MarginAllocationType.cross,
    )
    reporter = Reporter(market=market, exchange=exchange,
                        benchmark_symbols=benchmark_symbols, key=signal_setup.signal_key)

    backtest_setup_list.append(BacktestSetup(
        key=signal_setup.signal_key, signal_setup=signal_setup,
        portfolio=portfolio, exchange=exchange, reporter=reporter))

print("Portfolio is ready!")


In [ ]:

# RUN backtests
used_volume = 0
for backtest_setup in backtest_setup_list:
    market.reset()
    backtest = Backtester(market=market, portfolio=backtest_setup.portfolio,
                          exchange=backtest_setup.exchange, reporter=backtest_setup.reporter)

    # deposit some funds so we can trade
    symbol_volume = deposit_volume * \
        symbol_budgets[backtest_setup.signal_setup.symbol]
    volume = backtest_setup.signal_setup.budget_scale * symbol_volume
    used_volume = used_volume + volume
    backtest.exchange.transactions.add_deposit(
        asset=deposit_asset, volume=volume)
    backtest.run_all()
    backtest_setup.backtest = backtest
    print(
        f'Finished running the backtest {backtest_setup.key} with ' +
        f'{backtest_setup.signal_setup.budget_scale} * {deposit_volume} * {symbol_budgets[backtest_setup.signal_setup.symbol]} = ' +
        f'{volume} {deposit_asset} budget')


print(
    f'Finished running the all backtests! Total volume used {used_volume} {deposit_asset}')


In [ ]:
# Print Report
for backtest_setup in backtest_setup_list:
    # add additional fields to the plot
    meta_data = {"Algo": backtest_setup.key, "SL": backtest_setup.signal_setup.sl_percent *
                 100, "TP": backtest_setup.signal_setup.tp_percent * 100}
    additional_fields = pd.DataFrame({
        "Labels": list(meta_data.keys()),
        "Values": list(meta_data.values()),
    })
    
    # show plot graphs here
    if SHOW_PLOT_GRAPHS_SINGLE:
        backtest_setup.backtest.reporter.merged.plot(x="time_close", y=[
            f"{backtest_setup.key}__cumulative_returns",
            f"{benchmark_symbols[0]}__cumulative_returns"
        ], figsize=(30, 6))

        backtest_setup.backtest.reporter.merged.plot(x="time_close", y=[
            f"{backtest_setup.key}__drawdown",
            f"{benchmark_symbols[0]}__drawdown"
        ], figsize=(30, 6))

    # show report plot here
    if SHOW_HTML_REPORT_SINGLE:
        backtest_setup.backtest.reporter.show_report_plot(
            additional_fields=additional_fields)

    # save as an html file
    if SAVE_HTML_REPORT_SINGLE:
        base, quote = backtest_setup.signal_setup.symbol.split("/")
        filename = ''.join([f'{SAVE_HTML_REPORT_FOLDER}/xma-{base}-{quote}--',
                           f'{backtest_setup.signal_setup.interval}{backtest_setup.signal_setup.unit_of_time}-{backtest_setup.signal_setup.side.value}--',
                            f'(base-{backtest_setup.signal_setup.base_length}--trend-{backtest_setup.signal_setup.trend_length}--',
                            f'sl-{backtest_setup.signal_setup.sl_percent * 100}--tp-{backtest_setup.signal_setup.tp_percent * 100}',
                            f')--xma-trades--{date_from}--{date_to}.html'])
        backtest_setup.backtest.reporter.save_report_plot(
            additional_fields=additional_fields, filename=filename)


In [ ]:
# when we instantiate reporter for merging, we don't pass the exchange
merged_report = Reporter(
    market=market, benchmark_symbols=benchmark_symbols, key='algo')

# then we merge external snapshots (snapshots from strategy reporters)
snapshots = []
for backtest_setup in backtest_setup_list:
    snapshots.append(backtest_setup.reporter.raw_snapshots)

merged_report.merge_external_snapshots(snapshots)

# when we add trades
trades = []
for backtest_setup in backtest_setup_list:
    for ws in backtest_setup.portfolio.weighted_signals:
        trades = list(trades + ws.signal.get_trade_history())

merged_report.trades = list(map(lambda trade: trade.__dict__, trades))

# finally we generate report
merged_report.generate_report(resolution="1_day")

print('Report generated')

In [ ]:
# Print out
meta_data = {"Algo": "XMA 22", "Initial deposit": f"{deposit_volume} {deposit_asset}"}

for backtest_setup in backtest_setup_list:
    # add additional fields to the plot
    symbol_volume = deposit_volume * \
        symbol_budgets[backtest_setup.signal_setup.symbol]
    volume = backtest_setup.signal_setup.budget_scale * symbol_volume
    meta_data[f"{backtest_setup.key} - Initial deposit"] = f"{volume} {deposit_asset}"
    meta_data[f"{backtest_setup.key} - SL"] = backtest_setup.signal_setup.sl_percent * 100
    meta_data[f"{backtest_setup.key} - TP"] = backtest_setup.signal_setup.tp_percent * 100

additional_fields = pd.DataFrame({
    "Labels": list(meta_data.keys()),
    "Values": list(meta_data.values()),
})

# show plot graphs here
if SHOW_PLOT_GRAPHS_MERGED:
    merged_report.merged.plot(x="time_close", y=[
        "algo__cumulative_returns",
        f"{benchmark_symbols[0]}__cumulative_returns"
    ], figsize=(30, 6))

    merged_report.merged.plot(x="time_close", y=[
        "algo__drawdown",
        f"{benchmark_symbols[0]}__drawdown"
    ], figsize=(30, 6))

# show report plot here
if SHOW_HTML_REPORT_MERGED:
    merged_report.show_report_plot(additional_fields=additional_fields)

# save as an html file
if SAVE_HTML_REPORT_MERGED:
    base, quote = symbol.split("/")
    merged_report.save_report_plot(
        additional_fields=additional_fields,
        filename=f'{SAVE_HTML_REPORT_FOLDER}/xma-22--merged-report--{date_from}--{date_to}.html'
    )

if SAVE_AS_DATASET:
    merged_report.generate_report(resolution="1_hour")
    data = pd.DataFrame(merged_report.merged)
    
    data["close"] = data["algo__cumulative_returns"]
    data["open"] =  data["algo__cumulative_returns"].shift(1)
    
    # not taking care of high and low for now
    data["high"] = data["close"]
    data["low"] = data["close"]
    data["volume"] = 1
    
    MarketDataToTradingViewCSV(
        folder=SAVE_DATASET_FOLDER,        
        exchange="invao",
        symbol="XMA/USD",
        resolution="1_hour",
        df=data
    ).export_csv()
    